# 15 · Dense Retrieval（稠密检索）

> “问题向量化 → 向量库相似度 → Top-K”，RAG 检索的核心链路。

**本文件覆盖知识点**：Top-K / Similarity Threshold / Candidate Pool / Recall

```text
Query ──embed──> 向量 ──ANN──> Top-K chunk
```

In [ ]:
# 0) .env 配置 + 公共函数
from dotenv import load_dotenv; load_dotenv()
import os, numpy as np, faiss
from pathlib import Path
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def embed(texts, model='text-embedding-v3'):
    from dashscope import TextEmbedding
    if isinstance(texts, str): texts = [texts]
    r = TextEmbedding.call(model=model, input=texts, api_key=API_KEY)
    a = np.array([e['embedding'] for e in r.output['embeddings']], dtype='float32')
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-10)

def split_by_sent(text, n=160, ov=20):
    """极简切分（完整版在第7课）：按句号聚合到 n 字左右"""
    out, buf = [], ''
    for s in text.replace('\n', '').split('。'):
        if len(buf) + len(s) > n and buf:
            out.append(buf + '。'); buf = s
        else:
            buf += s + '。'
    if buf: out.append(buf)
    return out

# 1) 建一个小库（演示为主；量越大索引优势越明显）
texts = []
for p in sorted(Path('data').glob('*.md')):
    for piece in split_by_sent(p.read_text(encoding='utf-8')):
        texts.append((piece, p.name))  # (正文, 来源)
print('chunk 数:', len(texts))

vecs = embed([t[0] for t in texts])
idx = faiss.IndexFlatIP(vecs.shape[1]); idx.add(vecs)
print('索引就绪, 向量维度:', vecs.shape[1])

In [ ]:
# 2) Top-K + 阈值
def retrieve(query, k=3, threshold=-1.0):
    """稠密检索：返回 (text, source, score)；score<threshold 的被过滤"""
    qv = embed(query)
    sims, ids = idx.search(qv, k)
    out = []
    for i, s in zip(ids[0], sims[0]):
        if i == -1 or s < threshold:
            continue
        out.append((texts[i][0], texts[i][1], float(s)))
    return out

for q in ['星云客服机器人能私有化部署吗', '梅西进了多少球']:
    print('问题:', q)
    for t, src, s in retrieve(q, k=2):
        print(f'   {s:+.3f} [{src}] {t[:38]}')
    print()

## 关键概念

| 概念 | 含义 | 工程注意 |
|------|------|---------|
| **Top-K** | 取最像的 K 条 | 太小漏召回、太大进噪声；配合 Rerank(22) |
| **Candidate Pool(候选池)** | 先进 Top-50 粗池再精排 | 池太浅会把正确答案挡在外面 |
| **Similarity Threshold** | 相似度低于阈值丢弃 | 阈值过高误杀、过低漏放，靠评测调 |
| **Recall** | 该召回的召回了多少 | 检索层的核心质量指标(第33课) |

> 上面第二个无关问题命中分数明显偏低——正好演示阈值能挡住“文不对题”。

## 小结

- Dense Retrieval = 向量相似度取 Top-K；
- 用**候选池+阈值**控制召回质量；
- 稠密检索不懂精确关键词，需要**稀疏检索**互补 → 下一课。